In [3]:
from pathlib import Path

matches_file = Path("matches.csv")
if matches_file.exists():
    print("✅ matches.csv found!")
    print(f"   File size: {matches_file.stat().st_size / 1024:.1f} KB")
else:
    print("❌ matches.csv not found!")
    print("\nPlease either:")
    print("1. Run scraping-2.ipynb to generate matches.csv, OR")
    print("2. Copy matches.csv to this directory")
    print(f"\nCurrent directory: {Path.cwd()}")

✅ matches.csv found!
   File size: 273.2 KB


## Prerequisites

Before running this notebook, you need `matches.csv`. You can either:

1. **Run `scraping-2.ipynb`** in this directory to scrape fresh data from fbref.com (~10-15 minutes)
2. **Copy an existing matches.csv** file to this directory

Run the cell below to check if matches.csv exists:

# Premier League Match Predictor - Model Training

This notebook trains a Random Forest model to predict Premier League match outcomes and exports the necessary files for the web app:
- `model.joblib` - Trained model
- `feature_meta.json` - Feature metadata
- `features_reference.csv` - Latest team statistics for predictions

In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score
import joblib
import json
from pathlib import Path
from datetime import datetime

## 1. Load and Prepare Data

In [5]:
# Load matches data - update path to your actual matches.csv location
matches = pd.read_csv("matches.csv", index_col=0)
matches["Date"] = pd.to_datetime(matches["Date"])
print(f"Loaded {len(matches)} matches")
matches.head()

Loaded 1520 matches


,Date,Time,Comp,Round,Day,Venue,Result,GF,GA,Opponent,...,Match Report,Notes,Sh,SoT,Dist,FK,PK,PKatt,Season,Team
1,2023-08-11,20:00,Premier League,Matchweek 1,Fri,Away,W,3,0,Burnley,...,Match Report,NaN,17.0,8.0,13.9,0.0,0,0,2024,Manchester City
3,2023-08-19,20:00,Premier League,Matchweek 2,Sat,Home,W,1,0,Newcastle Utd,...,Match Report,NaN,14.0,4.0,17.9,0.0,0,0,2024,Manchester City
4,2023-08-27,14:00,Premier League,Matchweek 3,Sun,Away,W,2,1,Sheffield Utd,...,Match Report,NaN,29.0,9.0,17.3,2.0,0,1,2024,Manchester City
5,2023-09-02,15:00,Premier League,Matchweek 4,Sat,Home,W,5,1,Fulham,...,Match Report,NaN,6.0,4.0,14.8,0.0,1,1,2024,Manchester City
6,2023-09-16,15:00,Premier League,Matchweek 5,Sat,Away,W,3,1,West Ham,...,Match Report,NaN,29.0,13.0,16.4,1.0,0,0,2024,Manchester City


## 2. Feature Engineering

In [6]:
# Basic features
matches["venue_code"] = matches["Venue"].astype("category").cat.codes
matches["opponent_code"] = matches["Opponent"].astype("category").cat.codes
matches["hour"] = matches["Time"].str.replace(":.+", "", regex=True).astype("int")
matches["day_code"] = matches["Date"].dt.dayofweek
matches["target"] = (matches["Result"] == "W").astype("int")

In [7]:
# Rolling averages function
def rolling_averages(group, cols, new_cols):
    group = group.sort_values("Date")
    rolling_stats = group[cols].rolling(3, closed='left').mean()
    group[new_cols] = rolling_stats
    group = group.dropna(subset=new_cols)
    return group

In [8]:
# Create rolling average features
cols = ["GF", "GA", "Sh", "SoT", "Dist", "FK", "PK", "PKatt"]
new_cols = [f"{c}_rolling" for c in cols]

matches_rolling = matches.groupby("Team").apply(lambda x: rolling_averages(x, cols, new_cols))
matches_rolling = matches_rolling.droplevel("Team")
matches_rolling.index = range(matches_rolling.shape[0])

print(f"Created {len(new_cols)} rolling average features")
print(f"Final dataset: {len(matches_rolling)} matches")

Created 8 rolling average features
Final dataset: 1451 matches


## 3. Train Model

In [9]:
# Define predictors
predictors = ["venue_code", "opponent_code", "hour", "day_code"] + new_cols
print(f"Using {len(predictors)} features: {predictors}")

Using 12 features: ['venue_code', 'opponent_code', 'hour', 'day_code', 'GF_rolling', 'GA_rolling', 'Sh_rolling', 'SoT_rolling', 'Dist_rolling', 'FK_rolling', 'PK_rolling', 'PKatt_rolling']


In [10]:
# Split train/test
train = matches_rolling[matches_rolling["Date"] < '2023-01-01']
test = matches_rolling[matches_rolling["Date"] > '2023-01-01']

print(f"Train set: {len(train)} matches")
print(f"Test set: {len(test)} matches")

Train set: 268 matches
Test set: 1179 matches


In [11]:
# Train Random Forest
rf = RandomForestClassifier(n_estimators=50, min_samples_split=10, random_state=1)
rf.fit(train[predictors], train["target"])

print("Model trained successfully!")

Model trained successfully!


## 4. Evaluate Model

In [12]:
# Make predictions
preds = rf.predict(test[predictors])

# Calculate metrics
accuracy = accuracy_score(test["target"], preds)
precision = precision_score(test["target"], preds)

print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")

Accuracy: 0.621
Precision: 0.525


In [13]:
# Confusion matrix
combined = pd.DataFrame(dict(actual=test["target"], prediction=preds))
pd.crosstab(index=combined["actual"], columns=combined["prediction"])

prediction,0,1
actual,,
0,562,154
1,293,170


## 5. Prepare Reference Data for Predictions

Calculate the latest statistics for each team to use as reference data for making predictions.

In [14]:
# Get the latest rolling stats for each team
latest_stats = matches_rolling.sort_values('Date').groupby('Team').tail(1)

# Select relevant columns for predictions
reference_cols = ['Team'] + cols + new_cols
features_reference = latest_stats[reference_cols].reset_index(drop=True)

print(f"Reference data created for {len(features_reference)} teams")
features_reference.head()

Reference data created for 23 teams


,Team,GF,GA,Sh,SoT,Dist,FK,PK,PKatt,GF_rolling,GA_rolling,Sh_rolling,SoT_rolling,Dist_rolling,FK_rolling,PK_rolling,PKatt_rolling
0,Leicester City,2,1,13.0,4.0,17.5,0.0,0,0,1.000000,2.666667,7.000000,4.000000,15.100000,0.666667,0.333333,0.666667
1,Leeds United,1,4,19.0,2.0,15.8,0.0,0,0,1.333333,2.333333,8.000000,2.666667,14.200000,0.000000,0.000000,0.333333
2,Southampton,4,4,15.0,10.0,17.5,1.0,0,0,1.333333,3.000000,9.333333,2.000000,17.333333,0.333333,0.333333,0.333333
3,Newcastle United,4,2,12.0,7.0,15.4,2.0,0,0,2.333333,1.666667,20.333333,8.000000,16.000000,0.333333,0.000000,0.333333
4,Arsenal,2,1,26.0,5.0,14.1,0.0,0,0,2.333333,0.666667,14.666667,5.333333,13.900000,0.333333,0.333333,0.333333


## 6. Export Model and Data Files

In [15]:
# Create data directory if it doesn't exist
data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)

print(f"Exporting to: {data_dir.absolute()}")

Exporting to: /Users/deev/Documents/Premier League/backend/notebooks/../data


In [16]:
# 1. Export trained model
model_path = data_dir / "model.joblib"
joblib.dump(rf, model_path)
print(f"✓ Saved model to {model_path}")

✓ Saved model to ../data/model.joblib


In [17]:
# 2. Export feature metadata
feature_meta = {
    "feature_columns": predictors,
    "rolling_features": new_cols,
    "base_features": ["venue_code", "opponent_code", "hour", "day_code"],
    "rolling_window": 3,
    "rolling_cols": cols,
    "model_type": "RandomForestClassifier",
    "model_params": {
        "n_estimators": 50,
        "min_samples_split": 10,
        "random_state": 1
    },
    "trained_date": datetime.now().isoformat(),
    "train_samples": len(train),
    "test_accuracy": float(accuracy),
    "test_precision": float(precision),
    "target_classes": ["Loss/Draw", "Win"]
}

meta_path = data_dir / "feature_meta.json"
with open(meta_path, 'w') as f:
    json.dump(feature_meta, f, indent=2)
print(f"✓ Saved feature metadata to {meta_path}")

✓ Saved feature metadata to ../data/feature_meta.json


In [18]:
# 3. Export reference data
reference_path = data_dir / "features_reference.csv"
features_reference.to_csv(reference_path, index=False)
print(f"✓ Saved reference data to {reference_path}")

✓ Saved reference data to ../data/features_reference.csv


## 7. Verify Exports

In [19]:
# Verify all files exist
print("\n=== Verification ===")
print(f"Model file exists: {model_path.exists()}")
print(f"Metadata file exists: {meta_path.exists()}")
print(f"Reference data exists: {reference_path.exists()}")

if all([model_path.exists(), meta_path.exists(), reference_path.exists()]):
    print("\n✅ All files exported successfully!")
    print("\nYour FastAPI backend should now be able to load the model.")
    print("If the backend is running with --reload, it will automatically detect the changes.")
else:
    print("\n⚠️ Some files are missing!")


=== Verification ===
Model file exists: True
Metadata file exists: True
Reference data exists: True

✅ All files exported successfully!

Your FastAPI backend should now be able to load the model.
If the backend is running with --reload, it will automatically detect the changes.


## 8. Test Model Loading (Optional)

In [20]:
# Test loading the model back
loaded_model = joblib.load(model_path)
print("Model loaded successfully!")

# Test prediction
sample = test[predictors].iloc[0:1]
test_pred = loaded_model.predict(sample)
test_proba = loaded_model.predict_proba(sample)

print(f"\nSample prediction: {test_pred[0]}")
print(f"Probabilities: {test_proba[0]}")

Model loaded successfully!

Sample prediction: 1
Probabilities: [0.29103574 0.70896426]
